# 투자 리딩방/오픈채팅 유입 스팸 탐지 — 텔레그램 크롤링 → 이상치 탐지 파이프라인

이 노트북은 국내 SNS/텔레그램에서 유통되는 "무료 종목 리딩", "OO% 수익 인증" 류 게시물이
카카오 오픈채팅/텔레그램 초대 링크로 유도하는 패턴을 탐지하기 위한 파이프라인입니다.

**전체 흐름**
1. 텔레그램 공개 채널 크롤링 (Telethon)
2. 채널 확장 (스노우볼 샘플링)
3. 전처리/정제
4. 피처 엔지니어링 (텍스트 임베딩, 정형 패턴, 도메인 WHOIS, 근접중복 반복성)
5. 약지도 검증 라벨 조인 (금감원/더치트 등 신고 이력 — 학습 라벨이 아니라 평가용)
6. 이상치 탐지 모델 (Isolation Forest + 규칙 점수 앙상블)
7. 평가 (precision@K)
8. 시각화 및 결과 저장

> **주의사항**
> - 텔레그램 크롤링은 본인 명의 API 자격증명(`api_id`, `api_hash`, my.telegram.org 발급)으로,
>   **공개 채널만** 대상으로 진행하세요. 비공개 그룹 잠입/개인정보 대량 수집은 지양합니다.
> - 이 노트북은 기본적으로 `DEMO_MODE = True`로 설정되어 있어, 실제 텔레그램 크리덴셜 없이도
>   샘플 데이터로 4~8단계(피처 엔지니어링~이상치 탐지) 전체를 바로 실행/테스트할 수 있습니다.
> - 실제 크롤링을 하려면 `DEMO_MODE = False`로 바꾸고 본인 API 자격증명과 시드 채널을 입력하세요.


In [ ]:
!pip install -q telethon sentence-transformers datasketch python-whois scikit-learn

---
## 0단계. 실행 모드 설정

`DEMO_MODE = True`  → 크롤링을 건너뛰고 내장된 샘플 데이터로 파이프라인을 테스트
`DEMO_MODE = False` → 실제 Telethon 크롤링 수행 (본인 API 자격증명 필요)


In [ ]:
DEMO_MODE = True  # 실제 크롤링을 하려면 False로 변경하세요


---
## 1단계. 수집 (Telethon)

my.telegram.org 에서 발급받은 `api_id`/`api_hash`로 공개 채널 메시지를 수집합니다.
계정 생성일은 텔레그램 API로 얻을 수 없으므로, 대신 채널 구독자 수·프로필 사진 유무·
`views`/`forwards`(봇 증폭 여부 신호)를 함께 저장합니다.


In [ ]:
import json, time, asyncio
from getpass import getpass

RAW_PATH = "telegram_raw.jsonl"

if not DEMO_MODE:
    # Colab/Jupyter 커널은 이미 asyncio 이벤트 루프를 돌리고 있어서,
    # telethon.sync의 동기식 `with client:` 문법은 "You must use async with ..." 에러가 납니다.
    # (nest_asyncio로도 해결되지 않습니다 — Telethon이 run_until_complete를 시도하기도 전에
    #  loop.is_running()이면 바로 에러를 던지기 때문입니다.)
    # 따라서 진짜 async/await 문법으로 작성합니다. Colab/Jupyter는 셀에서 top-level await를 지원합니다.
    from telethon import TelegramClient

    api_id = int(getpass("api_id: "))
    api_hash = getpass("api_hash: ")
    client = TelegramClient("session_investment_spam", api_id, api_hash)

    async def crawl_channel(channel_username, out_path, limit=3000):
        entity = await client.get_entity(channel_username)
        with open(out_path, "a", encoding="utf-8") as f:
            async for msg in client.iter_messages(entity, limit=limit):
                if not msg.text:
                    continue
                record = {
                    "channel": channel_username,
                    "channel_id": entity.id,
                    "message_id": msg.id,
                    "date": msg.date.isoformat(),
                    "sender_id": msg.sender_id,
                    "text": msg.text,
                    "views": getattr(msg, "views", None),
                    "forwards": getattr(msg, "forwards", None),
                    "fwd_from": str(msg.fwd_from) if msg.fwd_from else None,
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        await asyncio.sleep(2)  # rate limit 완화
else:
    print("DEMO_MODE=True → 1단계 실제 크롤링은 건너뜁니다. (6단계 아래에서 샘플 데이터를 로드합니다)")


In [ ]:
# 실제 크롤링 시 아래 시드 채널 유저네임을 채워서 실행하세요.
SEED_CHANNELS = [
    # "example_channel_1",
    # "example_channel_2",
]

if not DEMO_MODE:
    async with client:
        for ch in SEED_CHANNELS:
            await crawl_channel(ch, RAW_PATH)
    print(f"수집 완료 → {RAW_PATH}")


---
## 2단계. 채널 확장 (스노우볼 샘플링)

수집된 텍스트에서 `t.me/...` 형태의 채널 링크를 추출해 크롤링 대상을 넓힙니다.
리딩방 생태계는 서로 홍보를 품앗이하는 경우가 많아, 2~3 hop만 돌려도
시드 대비 채널 수가 크게 늘어나는 것을 확인할 수 있습니다.


In [ ]:
import re

TME_PATTERN = re.compile(r"t\.me/(joinchat/[\w-]+|\+[\w-]+|[\w_]{5,})")

def extract_new_channels(text):
    return TME_PATTERN.findall(text or "")

if not DEMO_MODE:
    discovered = set()
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            discovered.update(extract_new_channels(rec["text"]))
    new_channels = discovered - set(SEED_CHANNELS)
    print(f"신규 발견 채널 수: {len(new_channels)}")
    # 필요 시 new_channels를 SEED_CHANNELS에 추가해 crawl_channel을 재실행하세요.


---
## 3단계. 데이터 로드 및 전처리

`DEMO_MODE`에 따라 실제 수집 결과(`telegram_raw.jsonl`) 또는 내장 샘플 데이터를 불러와
정규화 + 초대 링크/도메인 추출 + 정형 패턴(키워드) 매칭을 수행합니다.


In [ ]:
import pandas as pd

SAMPLE_DATA = [
    # ── 리딩방 스팸으로 의심되는 패턴 (여러 채널에서 유사 문구 반복) ──
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 1, "date": "2026-07-01T09:00:00",
     "sender_id": 5001, "text": "무료 종목 리딩 받아가세요! 이번주 수익률 인증 92% 선착순 카톡 오픈채팅 open.kakao.com/o/gAbC123",
     "views": 15000, "forwards": 320, "fwd_from": None},
    {"channel": "stock_free_2", "channel_id": 1002, "message_id": 1, "date": "2026-07-01T09:05:00",
     "sender_id": 5002, "text": "무료 종목 리딩방 오픈! 이번주 수익 인증 92% 선착순 마감 임박 open.kakao.com/o/gAbC123",
     "views": 14800, "forwards": 305, "fwd_from": None},
    {"channel": "stock_free_3", "channel_id": 1003, "message_id": 1, "date": "2026-07-01T09:10:00",
     "sender_id": 5003, "text": "급등주 무료 리딩 단톡방 초대 수익인증 92% 선착순 open.kakao.com/o/gAbC123",
     "views": 15200, "forwards": 340, "fwd_from": None},
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 2, "date": "2026-07-02T10:00:00",
     "sender_id": 5001, "text": "타점 잡아드립니다 무료 종목 리딩 단톡방 https://t.me/+xYzAbCd",
     "views": 9000, "forwards": 210, "fwd_from": None},
    {"channel": "stock_free_4", "channel_id": 1004, "message_id": 1, "date": "2026-07-03T11:00:00",
     "sender_id": 5004, "text": "선착순 무료 리딩방 수익 인증 92% open.kakao.com/o/gAbC123 서두르세요",
     "views": 16000, "forwards": 360, "fwd_from": None},
    # ── 정상적인 일반 대화/뉴스 공유로 추정되는 메시지 ──
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 1, "date": "2026-07-01T08:00:00",
     "sender_id": 6001, "text": "오늘 코스피 지수는 전일 대비 0.4% 상승 마감했습니다. 외국인 순매수 전환.",
     "views": 500, "forwards": 3, "fwd_from": None},
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 2, "date": "2026-07-02T08:00:00",
     "sender_id": 6001, "text": "금일 금통위 기준금리 동결 발표, 시장 예상과 부합.",
     "views": 480, "forwards": 2, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 1, "date": "2026-07-02T14:00:00",
     "sender_id": 6002, "text": "재무제표 분석 스터디 이번주 토요일 오후 2시에 진행합니다. 참여 원하시는 분은 댓글 남겨주세요.",
     "views": 120, "forwards": 1, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 2, "date": "2026-07-03T14:00:00",
     "sender_id": 6003, "text": "지난주 스터디 자료 공유드립니다. 링크는 곧 올리겠습니다.",
     "views": 110, "forwards": 0, "fwd_from": None},
    {"channel": "personal_diary_ch", "channel_id": 2003, "message_id": 1, "date": "2026-07-04T20:00:00",
     "sender_id": 6004, "text": "오늘 하루도 고생 많으셨습니다. 내일은 더 좋은 하루가 되길 바랍니다.",
     "views": 40, "forwards": 0, "fwd_from": None},
]

if DEMO_MODE:
    df = pd.DataFrame(SAMPLE_DATA)
else:
    rows = []
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)

print(f"로드된 메시지 수: {len(df)}")
df.head()


In [ ]:
INVITE_PATTERN = re.compile(r"(https?://)?(t\.me|open\.kakao\.com)/[^\s]+")
KEYWORD_PATTERNS = [
    r"무료\s*(리딩|종목)", r"수익\s*인증", r"선착순", r"\d+%\s*수익",
    r"단톡방", r"카톡\s*오픈채팅", r"급등주", r"타점",
]

def normalize(text):
    text = re.sub(r"[\u200b\uFE0F\u3164]", "", text)  # 제로폭/변형 문자 제거
    return text.strip()

def extract_invite_links(text):
    return [m.group(0) for m in INVITE_PATTERN.finditer(text or "")]

def extract_domain(link):
    m = re.search(r"(?:https?://)?([\w.-]+\.[a-zA-Z]{2,})", link)
    return m.group(1) if m else None

def keyword_hits(text):
    return sum(bool(re.search(p, text or "")) for p in KEYWORD_PATTERNS)

df["text_norm"] = df["text"].map(normalize)
df["invite_links"] = df["text_norm"].map(extract_invite_links)
df["invite_link_count"] = df["invite_links"].map(len)
df["domains"] = df["invite_links"].map(lambda links: [extract_domain(l) for l in links if extract_domain(l)])
df["kw_hits"] = df["text_norm"].map(keyword_hits)

df[["channel", "text_norm", "invite_link_count", "domains", "kw_hits"]]


---
## 4단계. 피처 엔지니어링

### (a) 텍스트 임베딩
한국어 문장 임베딩 모델로 텍스트를 벡터화하고 PCA로 차원을 축소합니다.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import numpy as np

embed_model = SentenceTransformer("jhgan/ko-sroberta-multitask")
embeddings = embed_model.encode(df["text_norm"].tolist(), show_progress_bar=False)

n_components = min(5, embeddings.shape[0] - 1, embeddings.shape[1])
pca = PCA(n_components=n_components, random_state=42)
embed_reduced = pca.fit_transform(embeddings)

for i in range(embed_reduced.shape[1]):
    df[f"emb_pca_{i}"] = embed_reduced[:, i]

print(f"임베딩 차원 {embeddings.shape[1]} → PCA {n_components}차원으로 축소")


### (b) 도메인 WHOIS 등록일

도메인이 최근 등록된 것일수록 스캠 사이트일 가능성이 높다는 가정으로 등록 후 경과일을 피처화합니다.
카카오 오픈채팅처럼 WHOIS 대상이 아닌 링크는 결측치로 남겨두고, 결측 여부 자체를 별도 피처로 둡니다.


In [ ]:
import whois
from datetime import datetime

_domain_age_cache = {}

def domain_age_days(domain):
    if domain in _domain_age_cache:
        return _domain_age_cache[domain]
    age = None
    try:
        w = whois.whois(domain)
        created = w.creation_date
        if isinstance(created, list):
            created = created[0]
        if created:
            age = (datetime.now() - created).days
    except Exception:
        age = None
    _domain_age_cache[domain] = age
    return age

def min_domain_age(domains):
    ages = [domain_age_days(d) for d in domains if d and "kakao.com" not in d and "t.me" not in d]
    ages = [a for a in ages if a is not None]
    return min(ages) if ages else None

df["domain_age_days"] = df["domains"].map(min_domain_age)
df["domain_age_missing"] = df["domain_age_days"].isna().astype(int)
df["domain_age_days"] = df["domain_age_days"].fillna(df["domain_age_days"].median() if df["domain_age_days"].notna().any() else 3650)

df[["channel", "domains", "domain_age_days", "domain_age_missing"]]


### (c) 반복성/근접중복 탐지 (MinHash LSH)

리딩방 스팸은 봇/총책이 템플릿 문구를 여러 채널에 복붙하는 경우가 많습니다.
서로 다른 채널·계정에서 문구가 얼마나 반복되는지를 MinHash 근접중복 탐지로 스코어화합니다.
정상적인 개인 발화는 거의 유일하지만, 스팸은 이 값이 비정상적으로 높게 나옵니다.


In [ ]:
from datasketch import MinHash, MinHashLSH

def get_minhash(text, num_perm=64):
    m = MinHash(num_perm=num_perm)
    for token in text.split():
        m.update(token.encode("utf8"))
    return m

lsh = MinHashLSH(threshold=0.6, num_perm=64)
duplicate_count = []

for idx, text in enumerate(df["text_norm"]):
    mh = get_minhash(text)
    matches = lsh.query(mh)
    duplicate_count.append(len(matches))
    lsh.insert(str(idx), mh)

df["duplicate_count"] = duplicate_count
df[["channel", "text_norm", "duplicate_count"]]


---
## 5단계. 약지도 검증 라벨 조인

금감원 불법금융광고 신고 데이터, 더치트, 경찰청 사이버캅 등에서 확보한 "신고 이력이 있는 도메인"
목록을 조인합니다. **이 라벨은 모델 학습용이 아니라 평가(precision proxy)용으로만 사용합니다.**
아래 `KNOWN_BAD_DOMAINS`는 예시이며, 실제 신고 데이터로 교체해서 사용하세요.


In [ ]:
KNOWN_BAD_DOMAINS = {
    "open.kakao.com/o/gAbC123",  # 예시: 금감원/더치트 등에서 신고 이력이 확인된 링크로 교체
}

def has_known_bad(domains_or_links, text):
    combined = " ".join(domains_or_links or [])
    return any(bad in combined or bad in (text or "") for bad in KNOWN_BAD_DOMAINS)

df["is_known_bad"] = [
    has_known_bad(links, text) for links, text in zip(df["invite_links"], df["text_norm"])
]

df[["channel", "invite_links", "is_known_bad"]]


---
## 6단계. 이상치 탐지 모델

피처 매트릭스를 구성하고 Isolation Forest로 통계적 이상 스코어를 산출한 뒤,
정형 패턴 기반 규칙 점수와 앙상블합니다.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def normalize01(series):
    s = series.astype(float)
    rng = s.max() - s.min()
    if rng == 0:
        return s * 0
    return (s - s.min()) / rng

emb_cols = [c for c in df.columns if c.startswith("emb_pca_")]
feature_cols = emb_cols + [
    "kw_hits", "invite_link_count", "duplicate_count",
    "domain_age_days", "domain_age_missing", "views", "forwards",
]

X = df[feature_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

contamination = min(0.3, max(0.05, 3 / len(df)))
clf = IsolationForest(n_estimators=300, contamination=contamination, random_state=42)
clf.fit(X_scaled)
df["anomaly_score"] = -clf.score_samples(X_scaled)

df["rule_score"] = (
    0.4 * normalize01(df["kw_hits"])
    + 0.4 * normalize01(df["duplicate_count"])
    + 0.2 * normalize01(df["invite_link_count"])
)

df["final_score"] = 0.5 * normalize01(df["anomaly_score"]) + 0.5 * df["rule_score"]

df.sort_values("final_score", ascending=False)[
    ["channel", "text_norm", "kw_hits", "duplicate_count", "domain_age_days", "anomaly_score", "rule_score", "final_score", "is_known_bad"]
]


---
## 7단계. 평가

`final_score` 상위 N개 중 `is_known_bad` 비율(precision proxy)을 확인합니다.
실제 프로젝트에서는 상위 N개를 사람이 직접 검수해 골드셋을 만들고 임계값을 튜닝하세요.


In [ ]:
TOP_N = min(5, len(df))
top_df = df.sort_values("final_score", ascending=False).head(TOP_N)

precision_at_n = top_df["is_known_bad"].mean()
print(f"Top-{TOP_N} 중 신고 이력(known_bad) 매칭 비율 (precision proxy): {precision_at_n:.2f}")

top_df[["channel", "text_norm", "final_score", "is_known_bad"]]


---
## 8단계. 시각화 및 결과 저장


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(df["final_score"], bins=10, edgecolor="black")
plt.axvline(df["final_score"].median(), color="red", linestyle="--", label="median")
plt.title("게시물 이상 점수(final_score) 분포")
plt.xlabel("final_score")
plt.ylabel("게시물 수")
plt.legend()
plt.tight_layout()
plt.show()

OUT_PATH = "telegram_spam_anomaly_scores.csv"
df.drop(columns=["invite_links", "domains"]).to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"결과 저장 완료 → {OUT_PATH}")


---
## 한계 및 유의사항

- `DEMO_MODE`의 샘플 데이터는 파이프라인 동작 확인용 예시일 뿐이며, 실제 텔레그램 데이터로 교체해야 유의미한 분석이 됩니다.
- WHOIS 조회는 도메인별로 rate limit이 있고, 카카오 오픈채팅 링크처럼 WHOIS 대상이 아닌 경우 결측치로 처리됩니다.
- `KNOWN_BAD_DOMAINS`는 금감원 불법금융광고 신고 데이터(공공데이터포털)·더치트·사이버캅 등에서 확보한 실제 리스트로 교체해야 하며,
  이는 학습 라벨이 아니라 평가용 검증셋으로만 사용해야 합니다(실제 신고 대비 전체 스캠의 극히 일부만 확인 가능하기 때문).
- 텔레그램 크롤링/저장 시 발신자 ID 등 개인정보가 포함될 수 있으므로, 장기 보관 전 익명화(해시 처리)를 권장합니다.
- `contamination` 값과 `final_score` 가중치(`0.5/0.5`)는 초기값이며, 실제 데이터로 상위 결과를 사람이 검수한 뒤 튜닝이 필요합니다.
